<a href="https://www.kaggle.com/code/ahmedfakhar123/ml-33-mini-batch-gradient-descent?scriptVersionId=343291718" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

---

# 🤖 ML 33 — Mini-Batch Gradient Descent

### *Understanding Mini-Batch GD, implementing it from scratch, and comparing it with SGD*

---



---

## ⚡ 1. Batch vs Stochastic vs Mini-Batch Gradient Descent

Before implementing Mini-Batch Gradient Descent, let's understand how it differs from the other two approaches.

### Batch Gradient Descent

Uses **all training samples** to calculate one gradient update.

```text
Entire Dataset
      ↓
Calculate Gradient
      ↓
Update Parameters
      ↓
Repeat
```

### Stochastic Gradient Descent

Uses **one sample** for each parameter update.

```text
One Sample
    ↓
Calculate Gradient
    ↓
Update Parameters
    ↓
Next Sample
```

### Mini-Batch Gradient Descent

Uses a **small group of samples** for each parameter update.

```text
Mini-Batch
   ↓
Calculate Gradient
   ↓
Update Parameters
   ↓
Next Mini-Batch
```

### 🧠 The idea

Mini-Batch Gradient Descent tries to combine the advantages of both:

| Method        | Samples per update | Updates | Stability     |
| ------------- | -----------------: | ------: | ------------- |
| Batch GD      |        All samples |     Few | High          |
| SGD           |                  1 |    Many | Low           |
| Mini-Batch GD |        Small batch |    Many | Moderate–High |

Mini-Batch GD is particularly important because this is the basic optimization strategy used extensively when training modern machine learning and deep learning models.



---

## 📦 2. What is Batch Size?

The **batch size** determines how many training samples are used to calculate each gradient update.

For example, if:

```text
Training samples = 331
Batch size = 32
```

then the model processes approximately:

```text
331 samples
   ↓
32 → 32 → 32 → ... → remaining samples
```

A smaller batch size means:

* More parameter updates
* More randomness
* Less memory usage

A larger batch size means:

* Fewer parameter updates
* More stable gradients
* More memory usage

---

## 📚 3. Import Required Libraries



In [1]:
from sklearn.datasets import load_diabetes

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

---

## 🩺 4. Load the Diabetes Dataset

We'll use Scikit-learn's Diabetes dataset.

It contains:

* **442 samples**
* **10 features**
* A numerical target representing disease progression


In [2]:
X, y = load_diabetes(return_X_y=True)
print(X.shape)
print(y.shape)

(442, 10)
(442,)


---

## ✂️ 5. Split the Dataset


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2)

---

# 📊 Part 1 — Linear Regression Baseline

Before implementing gradient descent, let's establish a baseline using Scikit-learn's `LinearRegression`.

## 6. Train Linear Regression



In [4]:
reg = LinearRegression()
reg.fit(X_train, y_train)

LinearRegression()

---

## 7. Inspect Model Parameters



In [5]:
print("Coefficients:")
print(reg.coef_)

print("\nIntercept:")
print(reg.intercept_)

Coefficients:
[ -36.49034836 -194.09811575  513.88593284  355.02971098 -891.00370348
  591.68794911  155.49417805  146.44668444  846.85202529   54.30219759]

Intercept:
152.65886927494057


---

## 8. Evaluate Linear Regression



In [6]:
y_pred = reg.predict(X_test)
r2_score(y_test, y_pred)

0.4429562235529033

---

# ⚙️ Part 2 — Mini-Batch Gradient Descent From Scratch

## 9. Implement `MBGDRegressor`

Now we'll implement Mini-Batch Gradient Descent using NumPy.

The key difference from our previous SGD implementation is that we calculate the gradient using **multiple samples at once**.

-batches
            for start in range(0, n_samples, self.batch_size):



In [7]:
import random

In [8]:
class MBGDRegressor:
    
    def __init__(self,batch_size,learning_rate=0.01,epochs=100):
        
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        
    def fit(self,X_train,y_train):
        
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        
        for i in range(self.epochs):
            
            for j in range(int(X_train.shape[0]/self.batch_size)):
                idx = random.sample(range(X_train.shape[0]),self.batch_size)
                
                y_hat = np.dot(X_train[idx],self.coef_) + self.intercept_
                
                intercept_der = -2 * np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)

                coef_der = -2 * np.dot((y_train[idx] - y_hat),X_train[idx])
                self.coef_ = self.coef_ - (self.lr * coef_der)
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_


---

## 🔍 Why Do We Shuffle the Data?

At the beginning of every epoch, we randomly shuffle the training samples:

```python
idx = np.random.permutation(n_samples)
```

This prevents the model from repeatedly seeing the training data in the same order.

Then we split the shuffled data into mini-batches.





---

# 🧪 Part 3 — Train Our Model

## 10. Create the Model

In [9]:
mbr = MBGDRegressor(batch_size=int(X_train.shape[0]/50), learning_rate=0.01, epochs=100)

---

## 11. Train the Model



In [10]:
mbr.fit(X_train,y_train)



---

## 12. Inspect Learned Parameters





In [11]:
print("Intercept:", mbr.intercept_)
print("Coefficients:", mbr.coef_)

Intercept: 153.82965944410356
Coefficients: [  24.80304897 -127.55415803  434.01897452  302.12069382  -17.1904513
  -73.79109087 -198.8332438   125.10607266  388.32021638  114.68000751]


---

## 13. Make Predictions


In [12]:
y_pred = mbr.predict(X_test)

---

## 14. Evaluate the Model


In [13]:
r2_score(y_test,y_pred)

0.4408396166281686

The exact value may vary because the training process involves random shuffling.



### 🧠 Important

There is **no universally best batch size**.

The optimal value depends on:

* Dataset size
* Learning rate
* Number of epochs
* Model architecture
* Available computational resources

Common batch sizes include:

```text
8, 16, 32, 64, 128, 256
```



---

# 🤖 Part 5 — Mini-Batch Training with Scikit-learn

Scikit-learn's `SGDRegressor` performs one-sample updates internally, but we can use its `partial_fit()` method to manually provide batches.


---

## 15. Create the SGD Model



In [14]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(learning_rate='constant',eta0=0.1)
batch_size = 35


---

## 16. Train Using Mini-Batches





In [15]:
for i in range(100):
    idx = random.sample(range(X_train.shape[0]),batch_size)
    sgd.partial_fit(X_train[idx],y_train[idx])

Here, `partial_fit()` updates the model using the batch we provide.

---

## 17. Inspect the Learned Parameters




In [16]:
print("Intercept:", sgd.intercept_)
print("Coefficients:", sgd.coef_)

Intercept: [148.83281616]
Coefficients: [  67.82022425  -45.36223505  336.93646037  259.6210484    27.50782434
  -13.03936243 -177.53181973  132.26070524  310.04179169  147.01718107]


---

## 18. Evaluate Scikit-learn's Model



In [17]:
y_pred = sgd.predict(X_test)
r2_score(y_test,y_pred)

0.4005374089948657


---

# 📊 Part 6 — Compare the Models

Let's compare our models with the Linear Regression baseline.


For your current experiment:

| Model             |     R² Score |
| ----------------- | -----------: |
| Linear Regression |   **0.4430** |
| Mini-Batch GD     | **≈ 0.4302** |
| Scikit-learn SGD  | **≈ 0.3302** |

## 📌 Interpretation

The Mini-Batch Gradient Descent implementation can get **very close to the Linear Regression solution**.

The Scikit-learn SGD result in this particular experiment is lower because the chosen learning rate, number of updates, and other optimization settings aren't necessarily optimal.

That doesn't mean SGD is inherently worse. It means **optimization hyperparameters matter**.



---

# 🧠 Batch Size — The Bigger Picture

We can visualize the three gradient descent approaches like this:

```text
                Training Dataset
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
       Batch GD       SGD       Mini-Batch GD
          │            │            │
      All samples    1 sample    Small batch
          │            │            │
          ↓            ↓            ↓
       1 update    Many updates  Many updates
```

### The trade-off

```text
Batch GD
Accuracy/Stability ↑
Updates ↓
Memory Usage ↑


SGD
Updates ↑
Noise ↑
Memory Usage ↓


Mini-Batch GD
Balance between both ⭐
```



---

# 🏁 Conclusion

In this notebook, we implemented **Mini-Batch Gradient Descent from scratch** and compared it with Linear Regression and Scikit-learn's SGD-based approach.

Mini-Batch Gradient Descent uses a **small subset of training samples for each parameter update**, providing a practical balance between the stability of Batch Gradient Descent and the speed of Stochastic Gradient Descent.

We also explored the importance of **batch size** and saw that optimization performance depends heavily on hyperparameters such as the **learning rate, batch size, and number of epochs**.

---

# 🚀 What's Next?

[**🤖 ML 34 — Polynomial Regression**](http://)

In the next notebook, we'll learn how **Polynomial Regression** can model non-linear relationships.

We'll cover:

* 📈 Linear vs Polynomial Regression
* 🧮 Polynomial Features
* 🛠️ Polynomial Regression from scratch
* 🤖 Scikit-learn implementation
* ⚠️ Underfitting vs Overfitting
* 📊 Model evaluation

[**Next:** **ML 34 — Polynomial Regression** 🚀](http://)

